<a href="https://colab.research.google.com/github/tiwariaadarsh-creator/Fine-Tuning-a-GPT-2-Model-for-a-Custom-Chatbot/blob/main/Fine_Tuning_a_GPT_2_Model_for_a_Custom_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Title : Fine-Tuning a GPT-2 Model for a Custom Chatbot**

# **Prepared By : Aadarsh Tiwari**



## **Project Overview:**

Welcome to this hands-on guide to fine-tuning a Large Language Model (LLM)! In this project, we will transform a general-purpose language model, the powerful GPT-2, into a specialized chatbot. Our goal is to leverage a dataset of 3,000 conversational pairs to teach the model a specific conversational style and knowledge base. This process, known as fine-tuning, is a cornerstone of modern NLP, allowing us to adapt massive pre-trained models for specific tasks without the prohibitive cost of training them from scratch. By the end of this notebook, you will have a functional chatbot customized with our data and a clear understanding of the end-to-end LLM fine-tuning workflow.

# **The Core Objective :**

Our primary objective is to demonstrate the power of transfer learning in NLP. We will take the `distilgpt2` model—a smaller, faster variant of GPT-2 that is perfect for experimentation in environments like Kaggle—and train it further on a custom question-and-answer dataset. We will then compare the responses of the original, base model with our newly fine-tuned model to concretely measure the improvement and see how the model's personality and knowledge have shifted.

# **Methodology at a Glance:**

Our approach is structured into a clear, step-by-step process:
1. Environment Setup: We will import essential libraries like PyTorch and Hugging Face's `transformers` and `datasets`.
2. Data Loading & Preparation: We will load the 3k conversations dataset and format it into a single text sequence per conversation, making it suitable for training a causal language model like GPT-2.
3. Model & Tokenizer Initialization: We will load the pre-trained `distilgpt2` model and its corresponding tokenizer, setting up the necessary configurations for training.
4. Fine-Tuning: Using the Hugging Face `Trainer` API, we will fine-tune the model on our prepared dataset. This is where the model learns the new conversational patterns.
5. Inference & Evaluation: Finally, we'll test our new chatbot! We will provide it with prompts and compare its generated answers against those from the original model to see our improvements in action.

# **Step 1: Environment Setup & Loading the Base Model/Tokenizer :**

## **Objective of This Step**:


In this step, we prepare our environment for working with Large Language Models using the Hugging Face transformers library alongside PyTorch. This includes installing any required dependencies, importing essential packages, and initializing the building blocks of our project — the pre-trained base model and its tokenizer. The tokenizer is responsible for converting human-readable text into a sequence of tokens (numerical IDs) that the model can understand, while the model itself is the neural network that processes those tokens to generate text predictions.


## **Why We Use DistilGPT-2:**

We will use distilgpt2, which is a distilled (smaller and faster) version of GPT-2. This choice allows us to train and experiment quickly in GPU environment without exhausting memory or exceeding time limits, while still benefiting from the rich language understanding learned during GPT-2's original training.

## **Actions We Will Perform :**

1. Import the necessary Python packages: transformers, torch, and optionally datasets.

2. Initialize the model distilgpt2 using the Hugging Face model hub.

3. Load the corresponding tokenizer so we can transform raw text into token IDs for the model to process.

4. Verify that the model and tokenizer are ready for fine-tuning on our custom dataset in the next steps.

In [1]:
# Install the Hugging Face Transformers library

!pip install transformers --quiet


# Import required libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the tokenizer for distilgpt2
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

# Load the base model (DistilGPT-2) for Causal Language Modeling
model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)

# Verify model and tokenizer are ready
sample_text = "Hello, how are you?"
encoded_input = tokenizer.encode(sample_text, return_tensors="pt").to(device)
print("Sample token IDs:", encoded_input)

Using device: cuda


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Sample token IDs: tensor([[15496,    11,   703,   389,   345,    30]], device='cuda:0')


In [2]:
import pandas as pd

# Dataset path

dataset_path = "/content/drive/MyDrive/Conversation.csv"

# Load CSV file into a Pandas DataFrame

df = pd.read_csv(dataset_path)

# Show basic info about the dataset

print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())

# Display first few rows

print("\nSample Rows:")
print(df.head())

# Check for missing values

print("\nMissing Values Per Column:")
print(df.isnull().sum())

# Show basic statistics (if numerical columns exist)

print("\nDataset Statistics:")
print(df.describe(include='all'))

Dataset Shape: (3725, 3)

Column Names: ['Unnamed: 0', 'question', 'answer']

Sample Rows:
   Unnamed: 0                             question  \
0           0               hi, how are you doing?   
1           1        i'm fine. how about yourself?   
2           2  i'm pretty good. thanks for asking.   
3           3    no problem. so how have you been?   
4           4     i've been great. what about you?   

                                     answer  
0             i'm fine. how about yourself?  
1       i'm pretty good. thanks for asking.  
2         no problem. so how have you been?  
3          i've been great. what about you?  
4  i've been good. i'm in school right now.  

Missing Values Per Column:
Unnamed: 0    0
question      0
answer        0
dtype: int64

Dataset Statistics:
         Unnamed: 0           question             answer
count   3725.000000               3725               3725
unique          NaN               3510               3512
top             NaN  wha

# **Step 2: Data Preparation for Fine-Tuning :**

## **Objective of This Step :**

The core objective here is to transform our structured dataset (with 'question' and 'answer' columns) into a format suitable for training a causal language model. GPT-2 learns by predicting the next token in a continuous sequence of text. Our current two-column format isn't a continuous sequence, so we must combine them into one. We will create a single text string for each row that clearly represents the conversational flow.

## **Our Formatting Strategy:**

To help the model learn the structure of a conversation, we will format each row into a single string like this:

 [Q]: How are you?

 [A]: I'm doing great, thanks!

Here, s and /s represent the start and end of a single conversational exchange. This special token (known as the EOS or End-of-Sequence token) is crucial because it teaches the model when a thought or response is complete, which is vital for generating coherent answers during inference.

## **Actions We Will Perform:**

1. Combine Columns: We will merge the 'question' and 'answer' columns into a new column, applying our formatting rule.

2. Configure Tokenizer: Since distilgpt2 does not have a default padding token, we will set its padding token to be the same as its end-of-sequence (EOS) token. This is a standard practice for this model.

3. Create a Dataset Object: We will convert our Pandas DataFrame into a Hugging Face Dataset object for efficient processing and tokenization.


In [3]:
# Step 2: Data Preparation for Fine-Tuning


from datasets import Dataset

# --- 1. Format the data into a single string per row ---
# We will format our data as "<s>[Q]: {question} [A]: {answer}</s>"
# <s> and </s> are special tokens that mark the start and end of a sequence.
# This helps the model understand the structure of a complete conversational turn.

def format_conversation(row):
    return f"<s>[Q]: {row['question']} [A]: {row['answer']}</s>"

# Apply the formatting to the DataFrame
df['text'] = df.apply(format_conversation, axis=1)

print("Sample of formatted data:")
print(df['text'].head())


# --- 2. Configure the tokenizer ---
# The distilgpt2 model doesn't have a default padding token.
# We'll use the end-of-sequence (EOS) token as our padding token.
# This is a common practice for GPT-style models.
tokenizer.pad_token = tokenizer.eos_token


# --- 3. Create a Hugging Face Dataset object ---
# The Hugging Face Trainer API works best with 'Dataset' objects.
# We will convert our formatted text column from the pandas DataFrame
# into this specialized format.

# First, create a new DataFrame with only the text we need
training_data = pd.DataFrame({'text': df['text']})

# Convert the pandas DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(training_data)

# --- 4. Tokenize the entire dataset ---
# We'll apply the tokenizer to all our text examples. The tokenizer will
# convert the text into numerical IDs that the model can understand.
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=['text'])

print("\nDataset has been processed and tokenized.")
print("A single example from the tokenized dataset:")
print(tokenized_dataset[0])

Sample of formatted data:
0    <s>[Q]: hi, how are you doing? [A]: i'm fine. ...
1    <s>[Q]: i'm fine. how about yourself? [A]: i'm...
2    <s>[Q]: i'm pretty good. thanks for asking. [A...
3    <s>[Q]: no problem. so how have you been? [A]:...
4    <s>[Q]: i've been great. what about you? [A]: ...
Name: text, dtype: object


Map:   0%|          | 0/3725 [00:00<?, ? examples/s]


Dataset has been processed and tokenized.
A single example from the tokenized dataset:
{'input_ids': [27, 82, 36937, 48, 5974, 23105, 11, 703, 389, 345, 1804, 30, 685, 32, 5974, 1312, 1101, 3734, 13, 703, 546, 3511, 30, 3556, 82, 29, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

# **Step 3: Fine-Tuning the Language Model :**

## **Objective of This Step :**

This is the core of our project. We will now take the pre-trained distilgpt2 model and train it further on our specially prepared conversational dataset. This process, known as fine-tuning, adjusts the model's internal weights to specialize its knowledge and conversational style to match our data. To manage the complexities of a training loop (like batching data, calculating loss, and updating model weights), we will use the high-level Trainer API from the Hugging Face library, which simplifies the entire process immensely.

## **Key Components: `TrainingArguments` and `Trainer` :**

The Trainer API requires two main components:

1. TrainingArguments: This is a configuration class where we define all the hyperparameters for our training run. This includes the number of epochs (how many times to go through the data), the batch size, the learning rate, and where to save the trained model.

2. Trainer: This is the main class that orchestrates the training. We simply provide it with our model, the training arguments, and our tokenized dataset. It handles the entire training loop automatically.

## **Actions We Will Perform :**

1. Define Training Arguments: We'll set up the training configuration, choosing parameters that are well-suited for a Kaggle environment.

2. Instantiate the Trainer: We will create an instance of the Trainer, passing it our model, dataset, and arguments.

3. Launch Training: We will call the .train() method to start the fine-tuning process. You will see the training loss decrease over time, indicating that the model is learning.

4. Save the Final Model: Once training is complete, we will save our newly fine-tuned model to a directory so we can use it for inference in the next step.

In [4]:
%pip install --upgrade transformers accelerate

In [5]:
import transformers
print(transformers.__version__)

5.19.0


In [6]:
# ===============================================
# Step 3: Fine-Tuning the Model (Corrected Code)
# ===============================================

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# --- 1. Define Training Arguments ---
# (This part remains exactly the same)

training_args = TrainingArguments(
    output_dir="./chatbot_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=500,
    save_strategy="epoch",
    report_to="none"
)

# --- 2. Create the Data Collator ---
# This collator will automatically create the 'labels' needed for the loss calculation.
# For GPT-2 style models (Causal LM), we set mlm=False.

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# --- 3. Instantiate the Trainer ---
# We now add the data_collator to the Trainer.

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,  # <-- This is the new key argument
)

# --- 4. Launch Training ---

print("Starting the fine-tuning process...")
trainer.train()
print("Fine-tuning complete.")

# --- 5. Save the Final Model and Tokenizer ---

trainer.save_model("./chatbot_model")
tokenizer.save_pretrained("./chatbot_model")

print("\nModel and tokenizer have been saved to './chatbot_model'")

Starting the fine-tuning process...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.955702
200,2.166938
300,2.045375
400,2.001744
500,2.035408
600,1.984974
700,1.981910
800,1.943343
900,1.915908
1000,1.895208


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model and tokenizer have been saved to './chatbot_model'


# **Step 4: Inference with the Fine-Tuned Chatbot Model :**

## **Objective of This Step :**

In this step, we load our fine-tuned GPT-2 variant (distilgpt2) from the ./chatbot_model directory and use it to generate chatbot responses to custom prompts. We will leverage the Hugging Face generate() method to produce coherent, context-aware text that reflects the conversational style learned during fine-tuning.

## **Key Actions to Perform :**

1. Load the saved model and tokenizer from ./chatbot_model.

2. Prepare a sample prompt that simulates a chatbot conversation (e.g., [Q]: How are you today?).

3. Tokenize the prompt and pass it to the model's generate() function to produce a continuation.

4. Decode the generated tokens back into text for human-readable output.

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import re

# 1. Load fine-tuned model & tokenizer

model_path = "./chatbot_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

# Helper function: Clean model output from special sequences

def clean_response(text, prompt):
    """
    This function removes the original prompt and special tokens
    from the generated text.
    """
    # Remove the original prompt part
    reply = text[len(prompt):]
    # Strip GPT special format tags and extra spaces
    reply = re.sub(r"<s>|</s>", "", reply)
    return reply.strip()

# 2. Define a list of questions to ask the bot

questions = [
    "Where are you today?",
    "what is your name?",
    "Do you feel Good?",
    "what are you looking for?",
    "what's your hobby?"
]

print("--- Starting Batch Inference ---")

# 3. Loop through the predefined questions and get answers

for user_q in questions:
    print(f"\n[Q]: {user_q}")

    # Format the prompt
    prompt = f"[Q]: {user_q} [A]:"

    # Tokenize the input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate a response from the model

    output_ids = model.generate(
        **inputs,
        max_length=100,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode the generated token IDs back to a string

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=False) # Keep special tokens for cleaning

    # Clean the response to show only the bot's answer

    reply = clean_response(generated_text, prompt)

    print(f"[Bot]: {reply}")

print("\n--- Batch Inference Finished ---")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

--- Starting Batch Inference ---

[Q]: Where are you today?
[Bot]: i'm having a party this friday.              <|endoftext|>

[Q]: what is your name?
[Bot]: i'm a painter.                            </

[Q]: Do you feel Good?
[Bot]: no, i don't feel bad about anything.           <|endoftext|>

[Q]: what are you looking for?
[Bot]: i'm looking for a job.    <|endoftext|>

[Q]: what's your hobby?
[Bot]: i enjoy painting.                            </s

--- Batch Inference Finished ---


# **Conclusion:**


In conclusion, this project demonstrates how a pre-trained GPT-2 language model can be fine-tuned to develop a custom chatbot capable of generating relevant and contextually appropriate responses. By preparing and preprocessing the training dataset, configuring the model's training parameters, and fine-tuning it on domain-specific conversational data, the model can adapt its responses to the intended use case.

The project provides practical experience in Natural Language Processing (NLP), text preprocessing, transfer learning, model training, and chatbot development using Python and the Hugging Face Transformers library. Fine-tuning helps the model learn patterns from the provided conversational data, improving its ability to generate responses that align with the target domain.

However, the chatbot's performance depends on the quality and quantity of the training data, the choice of hyperparameters, and the evaluation process. GPT-2 may sometimes generate repetitive, inaccurate, or irrelevant responses, so further improvements could include expanding the dataset, optimizing training parameters, evaluating response quality, and implementing a suitable conversational interface.

Overall, this project establishes a foundation for building customized conversational AI applications and demonstrates how pre-trained language models can be adapted to specific requirements through fine-tuning. Future enhancements could focus on improving response accuracy, maintaining conversational context, reducing repetitive outputs, and deploying the chatbot for real-world use.